## Dependencias

In [1]:
import pandas as pd
import numpy as np
import json
from collections import Counter

# Data cleaning
import re
import unicodedata

# Multiprocessing
from multiprocessing import Pool

# Visualización
import stylecloud
from IPython.display import Image

# Text mining
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from tensorflow.keras.preprocessing.text import Tokenizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.feature_extraction.text import TfidfVectorizer

from IPython.display import Image
from stylecloud import gen_stylecloud
from tensorflow.keras.utils import plot_model

# Modelo
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split,RandomizedSearchCV,GridSearchCV,KFold,StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, roc_auc_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import scipy.sparse as sp

# Para las etiquetas
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', 200)

c:\Users\azayas\Diplomado\myenv\Lib\site-packages\stylecloud\stylecloud.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


## Funciones

In [2]:
def clean_text(text, keep_twitter_tags=True):
    
    if not isinstance(text, str):
        return ""

    cleaned_text = text.lower()
    
    # Eliminamos los acentos pero nos quedamos con la ñ
    cleaned_text = ''.join(c for c in unicodedata.normalize('NFD', cleaned_text)
                           if unicodedata.category(c) != 'Mn')
    
    # Nos quedamos con las @, números, espacios, y #
    if keep_twitter_tags:
        pattern = r"[^a-zñ0-9@#]"
    else:
        pattern = r"[^a-zñ0-9 ]"
        
    cleaned_text = re.sub(pattern, " ", cleaned_text)
    
    cleaned_text = ' '.join(cleaned_text.split())
    
    return cleaned_text
    

def make_pool(func, params, threads):
    pool = Pool(threads)
    data = pool.map(func, params)
    pool.close()
    pool.join()
    del pool
    return data

def text_features(url, text):
    sid = SentimentIntensityAnalyzer()
    sentences = sent_tokenize(text)
    polarities = pd.DataFrame(map(sid.polarity_scores, sentences))
    words = pd.DataFrame(map(len, map(word_tokenize, sentences)))
    sentences_data = polarities.join(words).rename(columns={0: "n_words"})
    sentences_data["url"] = url
    return sentences_data

def get_wordcloud(text, icon="fas fa-comments", background_color=None, output_name="./wordcloud.png"):
    gen_stylecloud(text=text, icon_name=icon, background_color=background_color, output_name=output_name)
    return Image(filename=output_name)

def contador_palabras(df:pd.DataFrame, text_col:str, n_top:int):
    
    if df[text_col].dtype != 'O':
        print('El tipo de dato tiene que se string.')
        
    
    text_completo = ' '.join(df[text_col])
    
    palabras = text_completo.split()
    contador = Counter(palabras)
    
    top = contador.most_common(n_top)
    
    for palabra, frecuencia in top:
        print(f'{palabra}: {frecuencia}')

## Lectura de Datos

In [3]:
df = pd.read_csv(r"D:\Diplomado\MóduloIntegrador\Prácticas\PrácticaIII\datos\betsentiment-ES-tweets-sentiment-teams.csv", encoding='latin1')
df.head()

,tweet_date_created,tweet_id,tweet_text,language,sentiment,sentiment_score
0,2018-08-08T13:09:15.489000,1027179935184703489,"Alisson puede estar más tranquilo, no cargará ...",es,POSITIVE,"{""Neutral"":0.082259356975555419921875,""Negativ..."
1,2018-08-08T18:27:37.320000,1027260056092344320,@iPincheViky @ChelseaFC Es que el director eje...,es,NEUTRAL,"{""Neutral"":0.827011644840240478515625,""Negativ..."
2,2018-08-12T14:59:31.520000,1028657238116843520,Upto £100 #freebets &gt; https://t.co/cbjeMXI9...,es,NEUTRAL,"{""Neutral"":0.930982112884521484375,""Negative"":..."
3,2018-08-04T13:23:30.257000,1025733971320160257,"Bobby Duncan, primo de Steven Gerrard, deja la...",es,NEUTRAL,"{""Neutral"":0.906872212886810302734375,""Negativ..."
4,2018-07-28T11:21:06.480000,1023166450981396480,@TorreiraForeva @lepvtron @Arsenal @IntChampio...,es,NEUTRAL,"{""Neutral"":0.942405760288238525390625,""Negativ..."


## Extraction & Data processing

In [4]:
scores_df = df['sentiment_score'].apply(json.loads).apply(pd.Series)

In [5]:
df_ = pd.concat([
    df.drop(columns=['sentiment_score']),
    scores_df
], axis=1)

df_.head()

,tweet_date_created,tweet_id,tweet_text,language,sentiment,Neutral,Negative,Positive,Mixed
0,2018-08-08T13:09:15.489000,1027179935184703489,"Alisson puede estar más tranquilo, no cargará ...",es,POSITIVE,0.082259,0.036536,0.745924,0.135281
1,2018-08-08T18:27:37.320000,1027260056092344320,@iPincheViky @ChelseaFC Es que el director eje...,es,NEUTRAL,0.827012,0.078550,0.071943,0.022495
2,2018-08-12T14:59:31.520000,1028657238116843520,Upto £100 #freebets &gt; https://t.co/cbjeMXI9...,es,NEUTRAL,0.930982,0.031428,0.030184,0.007406
3,2018-08-04T13:23:30.257000,1025733971320160257,"Bobby Duncan, primo de Steven Gerrard, deja la...",es,NEUTRAL,0.906872,0.057496,0.024875,0.010757
4,2018-07-28T11:21:06.480000,1023166450981396480,@TorreiraForeva @lepvtron @Arsenal @IntChampio...,es,NEUTRAL,0.942406,0.031377,0.020383,0.005833


In [6]:
# Observamos que tenemos clases desbalanceadas

df_['sentiment'].value_counts(True)

sentiment
NEUTRAL     0.838946
POSITIVE    0.082920
NEGATIVE    0.071503
MIXED       0.006631
Name: proportion, dtype: float64

## Ingeniería de Datos

In [7]:
df_['tweet_text'].isna().sum()

np.int64(0)

In [8]:
# Quitamos los urls

df_['tweet_text'] = df_['tweet_text'].map(lambda x: re.sub('http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.#&+])+','', x ))

In [9]:
df_["len_tweet_text"] = df_['tweet_text'].str.len()
df_["n_words_tweet_text"] = df_['tweet_text'].str.split(" ").str.len()
df_["avg_len_words_tweet_text"] = df_['tweet_text'].str.split(" ").map(lambda x: np.mean([x for x in map(len, x)]))

In [10]:
contador_palabras(df_, 'tweet_text', 50)

de: 104691
el: 82869
la: 61831
en: 61350
que: 51971
y: 51844
a: 48771
del: 38346
con: 26591
?: 26382
por: 26149
los: 23556
al: 23420
??: 22958
un: 21536
no: 21453
se: 21172
para: 20637
es: 19219
El: 18148
lo: 14527
su: 12719
@ChelseaFC: 12485
@LFC: 12368
-: 11892
@ManUtd: 11802
|: 10065
una: 9796
como: 8031
#PremierLeague: 7905
las: 7772
más: 7695
le: 7587
pero: 7561
@Arsenal: 7415
@Everton: 6955
equipo: 6899
partido: 6286
@ManCity: 6188
ha: 6172
vs: 5875
#Chelsea: 5824
#Arsenal: 5792
ya: 5738
jugador: 5732
si: 5587
este: 5102
me: 5091
@premierleague: 5011
@SpursOfficial: 4949


In [11]:
stop_words = stopwords.words('spanish')

In [12]:
stop_words = [clean_text(word) for word in stop_words]
df_['tweet_text'] = df_['tweet_text'].map(clean_text).map(lambda text: [word for word in text.split() if word not in stop_words]).str.join(' ')
df_['tweet_text']

0         alisson puede tranquilo cargara peso ser arque...
1         @ipincheviky @chelseafc director ejecutivo muj...
2         upto 100 #freebets gt #lol #gunners #matchedbe...
3         bobby duncan primo steven gerrard deja cantera...
4         @torreiraforeva @lepvtron @arsenal @intchampio...
                                ...                        
132702    apunta diario mirror #manchesterunited pensand...
132703    @andresmarocco @sscnapoli @d ospina1 @arsenal ...
132704    final partido @everton 1 vs @westhamespanol 3 ...
132705    amistoso champions liga city perra liverpool #...
132706    @raul jimenez9 estreno goleador @wolves victor...
Name: tweet_text, Length: 132707, dtype: object

In [13]:
df_.head(3)

,tweet_date_created,tweet_id,tweet_text,language,sentiment,Neutral,Negative,Positive,Mixed,len_tweet_text,n_words_tweet_text,avg_len_words_tweet_text
0,2018-08-08T13:09:15.489000,1027179935184703489,alisson puede tranquilo cargara peso ser arque...,es,POSITIVE,0.082259,0.036536,0.745924,0.135281,152,29,4.275862
1,2018-08-08T18:27:37.320000,1027260056092344320,@ipincheviky @chelseafc director ejecutivo muj...,es,NEUTRAL,0.827012,0.078550,0.071943,0.022495,80,11,6.363636
2,2018-08-12T14:59:31.520000,1028657238116843520,upto 100 #freebets gt #lol #gunners #matchedbe...,es,NEUTRAL,0.930982,0.031428,0.030184,0.007406,86,11,6.909091


In [14]:
# La clase esra desbalanceada

df_['sentiment'].value_counts(True)

sentiment
NEUTRAL     0.838946
POSITIVE    0.082920
NEGATIVE    0.071503
MIXED       0.006631
Name: proportion, dtype: float64

# Modelado

In [15]:
num_palabras = df_['tweet_text'].str.lower().str.split().explode().nunique()

print(f'Hay {num_palabras} diferentes.')

Hay 100646 diferentes.


In [16]:
# Definición de variables

varc = ['len_tweet_text','n_words_tweet_text','avg_len_words_tweet_text']

txt_ = ['tweet_text']

um = ['tweet_id']

tgt = ['sentiment']

In [17]:
X = df_[txt_+varc].copy()

y = df_[tgt].copy()

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((106165, 4), (26542, 4), (106165, 1), (26542, 1))

In [19]:
X_train.reset_index(drop=True, inplace=True), X_test.reset_index(drop=True, inplace=True), y_train.reset_index(drop=True, inplace=True), y_test.reset_index(drop=True, inplace=True)

(None, None, None, None)

### Label Encoder

In [20]:
# Tratamos las etiqutas de la target

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train.values.ravel())

y_test_enc = le.transform(y_test.values.ravel())

### TF-IDF features

In [21]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    # min_df=0.001,
    # max_df=0.95,
    ngram_range=(1,2)
)

tfidf_train = vectorizer.fit_transform(X_train[txt_[0]])

tfidf_test = vectorizer.transform(X_test[txt_[0]])


### TAD

In [22]:
varc_sparse_train = sp.csr_matrix(X_train[varc].values)
varc_sparse_test = sp.csr_matrix(X_test[varc].values)

X_train_final = sp.hstack([tfidf_train, varc_sparse_train])
X_test_final = sp.hstack([tfidf_test, varc_sparse_test])

In [23]:
X_train[varc].shape, tfidf_train.shape, X_train.shape

((106165, 3), (106165, 10000), (106165, 4))

### Sample Weight

In [24]:
weights = compute_sample_weight(class_weight='balanced', y=y_train_enc)

weights

array([0.29801538, 0.29801538, 0.29801538, ..., 3.49272931, 3.49272931,
       0.29801538], shape=(106165,))

### XGBoost Classifier

In [25]:
xgb = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    n_jobs=-1,
    random_state=42
)

param_dist = {
    'n_estimators': [100, 200, 500],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 0.1, 0.2]
}

grid_search = GridSearchCV(estimator=xgb,
                           param_grid=param_dist,
                           cv=5,
                           scoring='f1_macro',
                           n_jobs=2)

### Entrenamiento

In [26]:
grid_search.fit(X_train_final, y_train_enc, sample_weight=weights)

KeyboardInterrupt: 

### Resultados

In [ ]:
print(f"Mejores Hiperparámetros:\n{grid_search.best_params_}")
print(f"Mejor F1 Macro (Validación): {grid_search.best_score_:.4f}")

Mejores Hiperparámetros:
{'subsample': 0.8, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1, 'gamma': 0.2, 'colsample_bytree': 0.8}
Mejor ROC AUC (Validación): 0.9974


In [ ]:
y_pred = grid_search.predict(X_test)
y_pred_proba = grid_search.predict_proba(X_test)

In [ ]:
print("Rendimiento en TEST:")
print(f"F1 Macro Final: {f1_score(np.ravel(y_test_enc), y_pred, average='macro'):.4f}")
print(classification_report(np.ravel(y_test_enc), y_pred))

Rendimiento en TEST:
ROC AUC Final: 1.0000
              precision    recall  f1-score   support

           0       0.93      0.99      0.96       190
           1       0.98      1.00      0.99      1890
           2       1.00      1.00      1.00     22274
           3       0.98      1.00      0.99      2188

    accuracy                           1.00     26542
   macro avg       0.97      1.00      0.98     26542
weighted avg       1.00      1.00      1.00     26542



### Guardamos el modelo

In [ ]:
import joblib

joblib.dump(grid_search.best_estimator_,'XGBClassifier_.pkl')

['XGBClassifier_.pkl']